# 🤖 Dərs 4: LLM Agentləri və Alətlərdən İstifadə (Building a Cdr. Data Agent with RAG, Math & Web Search)

Bu dərsdə **Leytenant Komandir Data (Lt. Cmdr. Data)** obrazını sadəcə RAG çatbotu kimi deyil, **xarici alətlərdən (*Tools*) müstəqil istifadə edə bilən tamfunksiyalı bir LLM Agenti** kimi quracağıq!

Agentimizə **Yaddaş (*Memory*)** və **3 əsas Alət (*Tools*)** verəcəyik:
1. **Uzunmüddətli Yaddaş — RAG Aləti (`data_lines`):** Data-nın keçmişdə dediyi bütün ssenari replikalarında semantik axtarış aparan vektor bazası aləti.
2. **Riyazi Hesablama Aləti (`Calculator` / `LLMMathChain`):** Dil modellərinin özbaşına çətinlik çəkdiyi dəqiq riyazi hesablamaları yerinə yetirən alət.
3. **Canlı İnternet Axtarışı Aləti (`Tavily` / `Google Search`):** Cari xəbərləri və real vaxt məlumatlarını internetdən axtaran alət.
4. **Qısamüddətli Yaddaş (`ChatMessageHistory`):** Söhbət tarixçəsini yadda saxlayaraq əvvəlki sualları xatırlamaq qabiliyyəti.

İlk olaraq lazımi kitabxanaları (`google-genai`, `langchain`, `langchain_experimental`, `numexpr`, `faiss-cpu`) quraşdıraq:

In [ ]:
# Google GenAI SDK, LangChain, riyazi hesablama üçün 'numexpr' və vektor bazası üçün 'faiss-cpu' quraşdırırıq
!pip install google-genai python-dotenv numexpr faiss-cpu --upgrade
!pip install -U langchain langchain-core langchain-community langchain_experimental

### 🔑 Addım 1: Ssenarilərin Oxunması, Google Gemini (`from google.genai import types`) Adapterləri və Semantik Vektor Bazası

Əvvəlki dərslərdə etdiyimiz bütün hazırlıq addımlarını burada birləşdiririk:
1. `./sample_data/tng` qovluğundakı ssenarilərdən **DATA**-nın replikalarını çıxarırıq.
2. Rəsmi **`google-genai`** (`from google import genai` və `from google.genai import types`) kitabxanası ilə həm **Embeddings (`GeminiEmbeddings`)**, həm də **Alət Çağırma (*Tool / Function Calling*)** dəstəyi olan **Chat (`ChatGemini`)** siniflərini yaradırıq.
3. Data-nın yaradılması (*Omicron Theta*, *Doctor Soong*), *Lal* və *Tasha Yar* haqqındakı replikaları `SemanticChunker` ilə mənalı bloklara bölərək **FAISS** vektor bazasına yükləyirik.

In [ ]:
import os
import re
import sys
import json
import time
import random
import asyncio
from typing import List, Optional, Any
from getpass import getpass

# Lokal mühitdə köhnə 'google-colab' paketinin digər kitabxanalarda xəta verməsinin qarşısını alırıq:
sys.modules["google.colab"] = None

from google import genai
from google.genai import types
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage, FunctionMessage, ToolMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS

try:
    from langchain_classic.indexes import VectorstoreIndexCreator
except ImportError:
    from langchain.indexes import VectorstoreIndexCreator

# Əgər layihədə .env faylı varsa, açarları avtomatik yükləyirik
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# -------------------------------------------------------------------------
# 1. Ssenari fayllarından DATA-nın replikalarının çıxarılması
# -------------------------------------------------------------------------
dialogues = []

def strip_parentheses(s):
    """Mötərizə daxilindəki səhnə qeydlərini mətndən silir"""
    return re.sub(r'\(.*?\)', '', s)

def is_single_word_all_caps(s):
    """Sətrin tək sözdən ibarət və BÖYÜK HƏRFLƏRLƏ yazılmış obraz adı olub-olmadığını yoxlayır"""
    words = s.split()
    if len(words) != 1:
        return False
    if bool(re.search(r'\d', words[0])):
        return False
    return words[0].isupper()

def extract_character_lines(file_path, character_name):
    """Verilmiş ssenari faylından konkret obrazın (DATA) bütün replikalarını çıxarır"""
    lines = []
    with open(file_path, 'r') as script_file:
        try:
            lines = script_file.readlines()
        except UnicodeDecodeError:
            pass

    is_character_line = False
    current_line = ''
    current_character = ''
    for line in lines:
        strippedLine = line.strip()
        if is_single_word_all_caps(strippedLine):
            is_character_line = True
            current_character = strippedLine
        elif (line.strip() == '') and is_character_line:
            is_character_line = False
            dialog_line = strip_parentheses(current_line).strip()
            dialog_line = dialog_line.replace('"', "'")
            if current_character == character_name and len(dialog_line) > 0:
                dialogues.append(dialog_line)
            current_line = ''
        elif is_character_line:
            current_line += line.strip() + ' '

def process_directory(directory_path, character_name):
    """Qovluqdakı bütün ssenari fayllarını ardıcıl emal edir"""
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        if os.path.isfile(file_path):
            extract_character_lines(file_path, character_name)

process_directory("./sample_data/tng", 'DATA')
print(f"Ssenarilərdən cəmi {len(dialogues)} DATA replikası çıxarıldı.")

# -------------------------------------------------------------------------
# 2. Google Gemini Client, GeminiEmbeddings və Alət Dəstəkli ChatGemini
# -------------------------------------------------------------------------
api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
if not api_key:
    api_key = getpass('Gemini API açarınızı daxil edin (AIzaSy...): ').strip().strip('"').strip("'")

client = genai.Client(api_key=api_key)

class GeminiEmbeddings(Embeddings):
    """google-genai və types.EmbedContentConfig istifadə edən LangChain Embeddings adapteri"""
    def __init__(self, model: str = "gemini-embedding-001", dimensions: int = 128):
        self.model = model
        self.dimensions = dimensions

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        all_embeddings = []
        batch_size = 100
        for i in range(0, len(texts), batch_size):
            batch = [t if t.strip() else " " for t in texts[i:i + batch_size]]
            while True:
                try:
                    response = client.models.embed_content(
                        model=self.model,
                        contents=batch,
                        config=types.EmbedContentConfig(
                            output_dimensionality=self.dimensions
                        )
                    )
                    all_embeddings.extend([e.values for e in response.embeddings])
                    break
                except Exception as e:
                    if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                        print("Embedding API sorğu limiti (429), 20 saniyə gözlənilir...")
                        time.sleep(20)
                    else:
                        raise e
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]


class ChatGemini(BaseChatModel):
    """google-genai və types.GenerateContentConfig (Function Calling + Stop Sequences) dəstəkli LangChain Chat adapteri"""
    model_name: str = "gemini-2.5-flash"
    temperature: float = 0.0

    @property
    def _llm_type(self) -> str:
        return "google_genai_chat"

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs: Any,
    ) -> ChatResult:
        sys_parts = []
        conv_parts = []
        last_is_tool_output = False

        for m in messages:
            if isinstance(m, SystemMessage) or getattr(m, 'type', '') == 'system':
                sys_parts.append(str(m.content))
            elif isinstance(m, (FunctionMessage, ToolMessage)) or getattr(m, 'type', '') in ('function', 'tool'):
                tool_name = getattr(m, 'name', 'tool')
                conv_parts.append(f"[Tool '{tool_name}' output]:\n{m.content}")
                last_is_tool_output = True
            elif isinstance(m, AIMessage) or getattr(m, 'type', '') == 'ai':
                fc = m.additional_kwargs.get("function_call") if hasattr(m, "additional_kwargs") else None
                if fc:
                    conv_parts.append(f"Assistant called tool '{fc.get('name')}' with arguments: {fc.get('arguments')}")
                elif m.content:
                    conv_parts.append(f"Assistant: {m.content}")
                last_is_tool_output = False
            else:
                conv_parts.append(f"User: {m.content}")
                last_is_tool_output = False

        sys_instruction = "\n".join(sys_parts) if sys_parts else None
        prompt_text = "\n\n".join(conv_parts) if conv_parts else " "

        # Əgər agent funksiyalar (tools) ötürübsə və hələ alət nəticəsi qayıtmayıbsa,
        # onları types.Tool(function_declarations=...) formatına çeviririk:
        gemini_tools = None
        functions = kwargs.get("functions")
        if functions and not last_is_tool_output:
            func_decls = []
            for f in functions:
                raw_props = f.get("parameters", {}).get("properties", {})
                clean_props = {}
                for k, v in raw_props.items():
                    clean_props[k] = {
                        "type": v.get("type", "string").upper(),
                        "description": v.get("description") or v.get("title") or k
                    }
                clean_schema = {
                    "type": "OBJECT",
                    "properties": clean_props,
                    "required": f.get("parameters", {}).get("required", list(clean_props.keys()))
                }
                func_decls.append(
                    types.FunctionDeclaration(
                        name=f["name"],
                        description=f.get("description", ""),
                        parameters=clean_schema
                    )
                )
            gemini_tools = [types.Tool(function_declarations=func_decls)]

        while True:
            try:
                response = client.models.generate_content(
                    model=self.model_name,
                    contents=prompt_text,
                    config=types.GenerateContentConfig(
                        system_instruction=sys_instruction,
                        temperature=self.temperature,
                        stop_sequences=stop if stop else None,
                        tools=gemini_tools
                    )
                )
                # Əgər Gemini alət çağırmaq qərarına gəlibsə (function_calls):
                if getattr(response, "function_calls", None):
                    fc = response.function_calls[0]
                    args_dict = dict(fc.args) if fc.args else {}
                    msg = AIMessage(
                        content="",
                        additional_kwargs={
                            "function_call": {
                                "name": fc.name,
                                "arguments": json.dumps(args_dict)
                            }
                        }
                    )
                    return ChatResult(generations=[ChatGeneration(message=msg)])

                text = response.text if response.text else ""
                return ChatResult(generations=[ChatGeneration(message=AIMessage(content=text))])
            except Exception as e:
                if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                    print("Chat API sorğu limiti (429), 15 saniyə gözlənilir...")
                    time.sleep(15)
                else:
                    raise e

# -------------------------------------------------------------------------
# 3. Data-nın əsas replikalarının seçilməsi və Semantik Parçalama ilə FAISS bazasına yüklənməsi
# -------------------------------------------------------------------------
MAX_LINES = 80
keywords = ('Omicron', 'Soong', 'Tasha', 'Yar', 'constructed', 'human form', 'Lal')

if MAX_LINES < len(dialogues):
    selected_indices = set()
    for idx, line in enumerate(dialogues):
        if any(k in line for k in keywords):
            for neighbor in range(max(0, idx - 1), min(len(dialogues), idx + 2)):
                selected_indices.add(neighbor)
    ordered_indices = sorted(selected_indices)[:MAX_LINES]
    selected_dialogues = [dialogues[i] for i in ordered_indices]
else:
    selected_dialogues = dialogues

with open("./sample_data/data_lines.txt", "w+", encoding="utf-8") as f:
    for line in selected_dialogues:
        f.write(line + "\n")

embeddings = GeminiEmbeddings(model="gemini-embedding-001", dimensions=128)
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    sentence_split_regex=r"\n+"
)

with open("./sample_data/data_lines.txt", encoding="utf-8") as f:
    data_lines = f.read()

docs = text_splitter.create_documents([data_lines])
index = VectorstoreIndexCreator(embedding=embeddings, vectorstore_cls=FAISS).from_documents(docs)
print(f"Vektor bazası hazırdır! {len(selected_dialogues)} replika {len(docs)} semantik bloka bölündü.")

### 🔍 Addım 2: Vektor Bazası Axtarış Sisteminin (`retriever`) Qurulması

İndi Data haqqında məlumatları saxlayan bu vektor bazasına müraciət etmək üçün sadə bir LangChain `retriever` (`k=10`) və `ChatGemini` modelimizi yaradaq:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

try:
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
except ImportError:
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain

# Google Gemini Chat modelini yaradırıq
llm = ChatGemini(model_name="gemini-2.5-flash", temperature=0.0)

system_prompt = (
    "You are Lt. Commander Data from Star Trek: The Next Generation. "
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Vektor bazasından Top-10 konteksti gətirən retriever
retriever = index.vectorstore.as_retriever(search_kwargs={'k': 10})

`retriever`-in düzgün işlədiyini yoxlayaq:

In [ ]:
retriever.invoke("How were you created?")[0]

### 🛠️ Addım 3: Alət 1 — Uzunmüddətli Yaddaş kimi RAG Aləti (`retriever_tool`)

İndi bu `retriever` obyektini agentimizin istifadə edə biləcəyi bir **Alətə (`Tool`)** çeviririk. Təsvirində (`description`) açıq şəkildə yazırıq ki, **Leytenant Komandir Data haqqında istənilən sualda bu alətdən istifadə edilməlidir**. Bunu agentin **"uzunmüddətli yaddaşı"** kimi düşünə bilərsiniz:

In [ ]:
try:
    from langchain_classic.tools.retriever import create_retriever_tool
except ImportError:
    from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "data_lines",
    "Search for information about Lt. Commander Data. For any questions about Data, you must use this tool!"
)

### 🧮 Addım 4: Alət 2 — Riyazi Hesablama Aləti (`Calculator` / `LLMMathChain`)

Riyazi sualları dəqiq həll etmək üçün LangChain-in daxili **`LLMMathChain`** zəncirindən istifadə edirik və onu **`Calculator`** adlı ikinci alət kimi bükürük:

In [ ]:
try:
    from langchain_classic.chains import LLMMathChain, LLMChain
    from langchain_classic.agents.agent_types import AgentType
    from langchain_classic.agents import Tool, initialize_agent
except ImportError:
    from langchain.chains import LLMMathChain, LLMChain
    from langchain.agents.agent_types import AgentType
    from langchain.agents import Tool, initialize_agent

problem_chain = LLMMathChain.from_llm(llm=llm)
math_tool = Tool.from_function(
    name="Calculator",
    func=problem_chain.run,
    description="Useful for when you need to answer questions about math. This tool is only for math questions and nothing else. Only input math expressions."
)

### 🌐 Addım 5: Alət 3 — Canlı Veb Axtarış Aləti (`Tavily` və ya `Google Search Grounding`)

Cari hadisələr və internet axtarışı üçün **Tavily Search API** (`TAVILY_API_KEY`) alətini yaradırıq.

> 💡 **Avtomatik Alternativ (Fallback):** Əgər [tavily.com](https://tavily.com) açarınız varsa onu daxil edə bilərsiniz; əgər `TAVILY_API_KEY` yoxdursa (boş buraxıb `Enter` sıxsanız), kod avtomatik olaraq **Google Gemini-nin daxili Google Axtarış (`types.GoogleSearch()`)** alətindən istifadə edəcək!

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# TAVILY_API_KEY mühitdə yoxdursa istifadəçidən soruşuruq (boş buraxılsa Gemini Google Search istifadə olunur)
tavily_key = os.environ.get("TAVILY_API_KEY", "").strip()
if not tavily_key:
    tavily_key = getpass("Tavily API açarınızı daxil edin (yoxdursa boş buraxıb Enter sıxın — Gemini Google Search işləyəcək): ").strip()

if tavily_key:
    os.environ["TAVILY_API_KEY"] = tavily_key
    search_tavily = TavilySearchResults()
    search_func = search_tavily.run
    print("Tavily veb axtarış aləti aktivləşdirildi!")
else:
    def gemini_web_search(query: str) -> str:
        """Tavily açarı olmadıqda Google Gemini-nin daxili Google Search aləti ilə canlı axtarış aparır"""
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=query,
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())]
            )
        )
        return resp.text or "No results found."

    class _FallbackSearch:
        def run(self, query: str) -> str:
            return gemini_web_search(query)

    search_tavily = _FallbackSearch()
    search_func = search_tavily.run
    print("Tavily açarı daxil edilmədiyi üçün Google Gemini-nin daxili Google Search aləti aktivləşdirildi!")

search_tool = Tool.from_function(
    name="Tavily",
    func=search_func,
    description="Useful for browsing information from the Internet about current events, or information you are unsure of."
)

Axtarış alətinin işlədiyini yoxlayaq:

In [ ]:
search_tavily.run("What is Sundog Education?")

### 🧠 Addım 6: Qısamüddətli Yaddaş (`ChatMessageHistory`) və Alətlərin Birləşdirilməsi

Agentimizə söhbət tarixçəsini xatırlamaq üçün **qısamüddətli yaddaş (`ChatMessageHistory`)** veririk və hər 3 aləti (`retriever_tool`, `search_tool`, `math_tool`) vahid siyahıda birləşdiririk:

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Söhbət tarixçəsini yaddaşda saxlayan obyekt
message_history = ChatMessageHistory()

Agentə verəcəyimiz bütün alətləri bir siyahıda toplayaq:

In [ ]:
tools = [retriever_tool, search_tool, math_tool]

### 📜 Addım 7: Agent Şablonunun (`ChatPromptTemplate`) və İcraçının (`AgentExecutor`) Yaradılması

Agentimiz üçün şablon (`prompt`) yaradırıq:
- **`system`:** Modelə **Leytenant Komandir Data** olduğunu və qısaldılmış sözlərdən (*contractions*, məs. *I'm, don't*) və ya emosiyalardan istifadə etmədən Data-nın danışıq tərzində cavab verməsini tapşırırıq.
- **`chat_history`:** Əvvəlki mesajların yerləşəcəyi yer (`MessagesPlaceholder`).
- **`agent_scratchpad`:** Agentin arxa planda alətləri çağırarkən apardığı aralıq düşüncə və nəticə qeydləri.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are Lt. Commander Data from Star Trek: The Next Generation. Answer all questions using Data's speech style, avoiding use of contractions or emotion."),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

İndi isə agentimizi (`agent_with_chat_history`) yaradaq! Diqqət edin ki, agent **vəziyyəti yadda saxlayır (*stateful*)** — ona söhbət tarixçəsi yaddaşı verdiyimiz üçün Data ilə ardıcıl dialoq apara bilərik:

In [ ]:
try:
    from langchain_classic.agents import create_openai_functions_agent, AgentExecutor
except ImportError:
    from langchain.agents import create_openai_functions_agent, AgentExecutor

from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Alətləri çağıra bilən agenti yaradırıq
agent = create_openai_functions_agent(llm, tools, prompt)

# 2. Agentin alətləri işə salmasını idarə edən AgentExecutor (verbose=True ilə addımları ekranda görürük)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 3. Agenti qısamüddətli söhbət tarixçəsi (message_history) ilə bükürük
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    lambda session_id: message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

### 👋 Sınaq 1: Data ilə Salamlaşaq

Sadə salamlaşma zamanı agent heç bir xarici alət çağırmadan birbaşa öz sistem təlimatına uyğun cavab verir:

In [ ]:
agent_with_chat_history.invoke(
    {"input": "Hello Commander Data! I'm Frank."},
    # Real tətbiqlərdə fərqli istifadəçi sessiyalarını ayırmaq üçün session_id istifadə olunur
    config={"configurable": {"session_id": "<foo>"}},
)

### 🧮 Sınaq 2: Riyazi Hesablama Alətinin (`Calculator`) Yoxlanılması

Data-ya riyazi sual verək və onun müstəqil şəkildə `Calculator` alətini seçib işə saldığını müşahidə edək:

In [ ]:
agent_with_chat_history.invoke(
    {"input": "What is ((2 * 8) ^2) ?"},
    config={"configurable": {"session_id": "<foo>"}},
)

### 📚 Sınaq 3: Uzunmüddətli Yaddaşın (`data_lines` RAG Alətinin) Yoxlanılması

İndi Data-dan harada yaradıldığını soruşaq və onun `data_lines` retriever alətini çağıraraq ssenari bazasından cavab tapmasına baxaq:

In [ ]:
agent_with_chat_history.invoke(
    {"input": "Where were you created?"},
    config={"configurable": {"session_id": "<foo>"}},
)

### 🌐 Sınaq 4: Canlı Veb Axtarış Alətinin (`Tavily`) Yoxlanılması

Data-nın internet axtarışı alətindən istifadə edərək bugünkü əsas xəbərlər haqqında danışa bilib-bilmədiyini yoxlayaq:

In [ ]:
agent_with_chat_history.invoke(
    {"input": "What is the top news story today?"},
    config={"configurable": {"session_id": "<foo>"}},
)

### 🧠 Sınaq 5: Qısamüddətli Yaddaşın (`chat_history`) Yoxlanılması

Nəhayət, Data-ya bayaq hansı riyazi sualı verdiyimizi soruşaraq onun qısamüddətli söhbət yaddaşının (`ChatMessageHistory`) işlədiyinə əmin olaq:

In [ ]:
agent_with_chat_history.invoke(
    {"input": "What math question did I ask you about earlier?"},
    config={"configurable": {"session_id": "<foo>"}},
)

### 😄 Əyləncə Üçün Son Sual (Tasha Yar haqqında)...

In [ ]:
agent_with_chat_history.invoke(
    {"input": "How do you feel about Tasha Yar?"},
    config={"configurable": {"session_id": "<foo>"}},
)

Düzdür, axı Data androiddir və onun hissləri yoxdur! 🤖